# Day 5 — Vector Databases + FAISS from Scratch
---
> **Phase:** 1 — LLM Foundations  
> **Status:** Complete

## Key Concepts

### 1. Why Numpy Fails at Scale

Problem 1 — No persistence. Vectors live in memory only. Script ends, all vectors gone. Next run, re-embed everything from scratch. 100,000 documents = hours of recomputation every run.

Problem 2 — No scalability. Computes similarity against every document for every query. 1 million documents = 1 million comparisons per search.

Problem 3 — No metadata. Numpy stores only vectors, nothing else.

### 2. What FAISS Adds

| Numpy Array | FAISS Index |
|------------|-------------|
| Memory only | Saveable to disk |
| Lost on exit | Reloadable instantly |
| Exact search | Approximate search (faster) |
| Python speed | C++ speed under the hood |

### 3. L2 Distance vs Cosine Similarity

L2 Distance measures straight line distance between vector tips. Affected by both direction and magnitude. Lower score = more similar. Filter: keep results BELOW threshold.

Cosine Similarity measures angle between vectors. Affected by direction only, magnitude ignored. Higher score = more similar. Filter: keep results ABOVE threshold.

Production fix: Normalize vectors to length 1. L2 and cosine become mathematically equivalent.

### 4. FAISS Core API

```python
index = faiss.IndexFlatL2(384)                       # dimension must match model
index.add(vectors)                                   # vectors must be float32
index.ntotal                                         # count of stored vectors
distances, indices = index.search(query_vector, k)  # search top k
faiss.write_index(index, path)                       # save to disk (binary)
index = faiss.read_index(path)                       # load from disk
```

### 5. Persistence Pattern

```python
if os.path.exists(index_path):
    index = faiss.read_index(index_path)         # instant load
else:
    vectors = model.encode(docs).astype(np.float32)
    index = faiss.IndexFlatL2(vectors.shape[1])
    index.add(vectors)
    faiss.write_index(index, index_path)         # save once
```

100,000 documents takes 3 hours to embed. With this pattern, pay that cost ONCE. Every subsequent run loads in milliseconds.

### 6. OS Module

```python
os.getcwd()                                      # current directory
os.path.join('Day-5', 'file.index')              # safe path building
os.path.exists('file.index')                     # True or False
os.path.dirname(os.path.abspath(__file__))       # script folder

# Production pattern always use this
script_dir = os.path.dirname(os.path.abspath(__file__))
file_path  = os.path.join(script_dir, 'file.index')
```

### 7. zip()

```python
distances = [0.98, 1.62, 1.63]
indices   = [0,    6,    2   ]

for dist, idx in zip(distances, indices):
    print(dist, idx)
# 0.98  0
# 1.62  6
# 1.63  2
```

Stitches two lists into pairs. No index counter needed. Clean and readable.

### 8. Critical Shape Rule

```python
model.encode(query)    # shape (384,)    WRONG for FAISS
model.encode([query])  # shape (1, 384)  CORRECT for FAISS
"FAISS actually expects a batch format, even when you're searching with just one query. That's why (1, 384) is correct — it's a batch of 1. If you passed (384,) it would reject it because it always expects a 2D matrix."
# Always wrap single query in a list
```

### 9. Semantic Gap

Query: 'is it worth the price?' did not match 'Price is too high for the specs you get'. Same topic, different angle. Embedding model placed them too far apart. Solution: Advanced RAG techniques in Phase 2 — query rewriting, HyDE.

In [ ]:
# Day 5 Complete Persistent Search System
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
import os

model = SentenceTransformer('all-MiniLM-L6-v2')

documents = [
    'The battery life on this laptop is incredible, lasts all day',
    'Screen resolution is disappointing, colors look washed out',
    'Keyboard feels premium and typing experience is excellent',
    'The laptop runs very hot during heavy tasks, fan is loud',
    'Lightweight and portable, perfect for travel and commuting',
    'RAM is soldered and cannot be upgraded, big disappointment',
    'Boot time is blazing fast, SSD performance is outstanding',
    'Webcam quality is terrible, grainy even in good lighting',
    'Build quality feels solid, no flex in the chassis at all',
    'Price is too high for the specs you get with this machine'
]

script_dir = os.path.dirname(os.path.abspath(__file__))
index_path = os.path.join(script_dir, 'reviews.index')

if os.path.exists(index_path):
    print('Loading existing index from disk...')
    loaded_index = faiss.read_index(index_path)
    print(f'Index loaded. Vectors: {loaded_index.ntotal}')
else:
    print('No index found. Building from scratch...')
    document_vectors = model.encode(documents).astype(np.float32)
    dimension = document_vectors.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(document_vectors)
    faiss.write_index(index, index_path)
    loaded_index = index
    print(f'Index built and saved. Vectors: {loaded_index.ntotal}')


def search(query, top_k=3, distance_threshold=1.2):
    query_vector = model.encode([query]).astype(np.float32)
    distances, indices = loaded_index.search(query_vector, top_k)

    print(f"\nQuery: '{query}'")
    print('-' * 50)

    results = [
        (dist, idx) for dist, idx in zip(distances[0], indices[0])
        if dist <= distance_threshold
    ]

    if not results:
        print('No relevant results found.')
    else:
        for i, (dist, idx) in enumerate(results):
            print(f'  Rank {i+1} | Distance: {dist:.4f} | {documents[idx]}')


print("Search ready. Type quit to exit.")
while True:
    query = input('Enter query: ')
    if query.strip().lower() == 'quit':
        break
    search(query)

## Observations

| Query | Result | Distance | Lesson |
|-------|--------|----------|--------|
| How long does battery last? | Battery life review | 0.9869 | Strong semantic match |
| Is the screen good? | Screen resolution review | 1.0965 | Different framing, still matched |
| How is the keyboard? | Keyboard review | 0.6262 | Very strong match |
| Is it worth the price? | No results | above 1.2 | Semantic gap |
| Free charger? | No results | above 1.2 | Correctly filtered |

## Summary

Why numpy fails at scale. FAISS and what it solves. IndexFlatL2. L2 distance vs cosine similarity. Saving and loading index to disk. OS module path handling. Distance threshold. Persistence pattern. zip(). Shape rule for single query. Debugging file path errors. Semantic gap.

## Coming in Day 6

LangChain Fundamentals. You now understand what every abstraction does under the hood. LangChain will feel obvious, not magical.